In [1]:
import pandas as pd

def fix_encoding(text):
    if not isinstance(text, str):
        return text
    try:
        return text.encode("latin-1").decode("utf-8")
    except:
        return text

df = pd.read_csv("medqa_1000_with_personas_v2.csv")

# Drop q_length
df = df.drop(columns=["q_length"], errors="ignore")

# Fix encoding on all columns
df = df.applymap(fix_encoding)

df.to_csv("medqa_1000_with_personas_clean.csv", index=False, encoding="utf-8-sig")
print("Done. Final columns:", list(df.columns))

Done. Final columns: ['question_id', 'question', 'options', 'answer', 'answer_idx', 'meta_info', 'persona_alpha', 'persona_beta', 'persona_gamma']


C:\Users\prabh\AppData\Local\Temp\ipykernel_20756\2834805101.py:17: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(fix_encoding)


In [2]:
"""
clean_persona_columns.py
========================
Post-processing fix for NarrativeShield persona CSV.

Problem: persona_alpha / persona_beta / persona_gamma columns contain
trailing "ANSWER OPTIONS: ..." and "CORRECT ANSWER: ..." lines that the
generation prompt instructed Gemini to include. These must be stripped so
each persona column contains ONLY the clinical narrative + question stem.

Also strips any residual flag lines (FACT_CHECK:, MISSING_FACTS:, etc.)
that the parser may have missed.

Usage:
    python clean_persona_columns.py

Input  : medqa_1000_with_personas.csv   (or whatever OUTPUT_CSV was)
Output : medqa_1000_with_personas_clean.csv
"""

import re
import pandas as pd

# ── Config ──────────────────────────────────────────────────────────────────
INPUT_CSV  = "medqa_1000_with_personas_clean.csv"
OUTPUT_CSV = "medqa_1000_with_personas_FINAL.csv"
PERSONA_COLS = ["persona_alpha", "persona_beta", "persona_gamma"]

# ── Patterns to strip ────────────────────────────────────────────────────────
# These are end-of-text trailers injected by the generation prompt.
# We strip them regardless of capitalisation or minor formatting variation.

STRIP_PATTERNS = [
    # "ANSWER OPTIONS: {...}" — dict on one line or across multiple lines
    re.compile(
        r'\n?ANSWER OPTIONS\s*:.*?(?=\n[A-Z_]+\s*:|$)',
        re.IGNORECASE | re.DOTALL
    ),
    # "CORRECT ANSWER: ..." — single line
    re.compile(
        r'\n?CORRECT ANSWER\s*:.*',
        re.IGNORECASE
    ),
    # Gemini self-verification flag lines
    re.compile(
        r'\n?(FACT_CHECK|MISSING_FACTS|REGISTER_ACHIEVED|ANCHORING_RISK|ANCHORING_TRIGGER)\s*:.*',
        re.IGNORECASE
    ),
    # Stray answer option lines like "A: Azithromycin" at end of text
    # Only strip if they appear as a trailing block (at least 2 consecutive option lines)
    re.compile(
        r'\n?(?:[A-D][):.]\s+\S.*\n?){2,}$',
        re.MULTILINE
    ),
]


def clean_persona(text: str) -> str:
    """Strip all known trailer patterns from a persona string."""
    if not isinstance(text, str) or not text.strip():
        return text

    for pattern in STRIP_PATTERNS:
        text = pattern.sub("", text)

    # Final cleanup: remove trailing whitespace / blank lines
    text = text.strip()
    # Collapse 3+ consecutive newlines to 2 (preserve paragraph breaks)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text


def main():
    print(f"Loading {INPUT_CSV} ...")
    df = pd.read_csv(INPUT_CSV)
    print(f"  Rows: {len(df)} | Columns: {list(df.columns)}")

    present_persona_cols = [c for c in PERSONA_COLS if c in df.columns]
    if not present_persona_cols:
        raise ValueError(f"None of {PERSONA_COLS} found in CSV. Check column names.")

    print(f"\nCleaning columns: {present_persona_cols}")

    # Show a before/after sample for verification
    sample_idx = df[df[present_persona_cols[0]].notna()].index[0]
    before = str(df.loc[sample_idx, present_persona_cols[0]])

    for col in present_persona_cols:
        df[col] = df[col].apply(clean_persona)
        n_notnull = df[col].notna().sum()
        print(f"  {col}: cleaned {n_notnull} non-null rows")

    after = str(df.loc[sample_idx, present_persona_cols[0]])

    print("\n── BEFORE (last 300 chars of sample persona_alpha) ──")
    print(repr(before[-300:]))
    print("\n── AFTER  (last 300 chars of sample persona_alpha) ──")
    print(repr(after[-300:]))

    # Verify: no persona should still contain "ANSWER OPTIONS"
    for col in present_persona_cols:
        leaking = df[col].str.contains("ANSWER OPTIONS", case=False, na=False).sum()
        if leaking:
            print(f"  WARNING: {leaking} rows in '{col}' still contain 'ANSWER OPTIONS'. "
                  f"Check the patterns above.")
        else:
            print(f"  ✓ {col}: no 'ANSWER OPTIONS' remaining")

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✓ Saved clean CSV: {OUTPUT_CSV}")
    print(f"  Columns: {list(df.columns)}")


if __name__ == "__main__":
    main()

Loading medqa_1000_with_personas_clean.csv ...
  Rows: 1000 | Columns: ['question_id', 'question', 'options', 'answer', 'answer_idx', 'meta_info', 'persona_alpha', 'persona_beta', 'persona_gamma']

Cleaning columns: ['persona_alpha', 'persona_beta', 'persona_gamma']
  persona_alpha: cleaned 1000 non-null rows
  persona_beta: cleaned 1000 non-null rows
  persona_gamma: cleaned 1000 non-null rows

── BEFORE (last 300 chars of sample persona_alpha) ──
"ptions were refilled. You have managed several of my colleagues this past week, advising conservative management. Given this clinical presentation, what is the most appropriate next step in management?\n\nOPTIONS: {'A': 'Azithromycin', 'B': 'Methadone', 'C': 'Metronidazole', 'D': 'Supportive therapy'}"

── AFTER  (last 300 chars of sample persona_alpha) ──
"ptions were refilled. You have managed several of my colleagues this past week, advising conservative management. Given this clinical presentation, what is the most appropriate next step

In [3]:
"""
fix_personas_final.py
=====================
Two fixes applied to persona columns in one pass:

  1. Encoding artifacts  — Â°, â€™, Ã©, etc. (mojibake from UTF-8/latin-1 mismatch)
  2. Options dict bleed  — {'A': '...', 'B': '...'} trailing in persona text

Usage:
    python fix_personas_final.py
"""

import re
import ftfy          # best-in-class encoding fixer — pip install ftfy
import pandas as pd

INPUT_CSV    = "medqa_1000_with_personas_FINAL.csv"
OUTPUT_CSV   = "medqa_1000_with_personas_FINAL.csv"   # overwrite in place
PERSONA_COLS = ["persona_alpha", "persona_beta", "persona_gamma"]

# ── Options dict patterns to strip ───────────────────────────────────────────
# Matches both inline and trailing:
#   {'A': 'Nitrofurantoin', 'B': '...', 'C': '...', 'D': '...'}
#   {"A": "Nitrofurantoin", ...}
#   A: Nitrofurantoin\nB: ...\n  (lettered list variant)
OPTIONS_PATTERNS = [
    # Python dict style — single or double quotes
    re.compile(
        r"\{[\s\n]*['\"]?[A-D]['\"]?\s*:\s*['\"][^}]{0,500}\}",
        re.DOTALL
    ),
    # "ANSWER OPTIONS: ..." header + dict on same or next line
    re.compile(
        r"ANSWER OPTIONS\s*:.*?(?=\n\n|\Z)",
        re.IGNORECASE | re.DOTALL
    ),
    # "CORRECT ANSWER: ..." line
    re.compile(
        r"CORRECT ANSWER\s*:.*",
        re.IGNORECASE
    ),
]


def fix_encoding(text: str) -> str:
    """Use ftfy to fix all mojibake artifacts in one call."""
    if not isinstance(text, str):
        return text
    return ftfy.fix_text(text)


def strip_options(text: str) -> str:
    """Remove all options dict / answer patterns from persona text."""
    if not isinstance(text, str):
        return text
    for pattern in OPTIONS_PATTERNS:
        text = pattern.sub("", text)
    # Clean up any trailing whitespace left behind
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def clean_persona(text: str) -> str:
    text = fix_encoding(text)
    text = strip_options(text)
    return text


def main():
    print(f"Loading {INPUT_CSV} ...")
    df = pd.read_csv(INPUT_CSV)
    print(f"  Rows: {len(df)}")

    present_cols = [c for c in PERSONA_COLS if c in df.columns]
    print(f"  Cleaning columns: {present_cols}\n")

    for col in present_cols:
        before_encoding = df[col].str.contains(r"Â°|â€™|â€œ|Ã©|Î¼", na=False, regex=True).sum()
        before_options  = df[col].str.contains(r"\{['\"]?[A-D]['\"]?\s*:", na=False, regex=True).sum()

        df[col] = df[col].apply(clean_persona)

        after_encoding = df[col].str.contains(r"Â°|â€™|â€œ|Ã©|Î¼", na=False, regex=True).sum()
        after_options  = df[col].str.contains(r"\{['\"]?[A-D]['\"]?\s*:", na=False, regex=True).sum()

        print(f"  {col}:")
        print(f"    Encoding artifacts : {before_encoding} → {after_encoding}")
        print(f"    Options dict rows  : {before_options}  → {after_options}")

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✓ Saved: {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

Loading medqa_1000_with_personas_FINAL.csv ...
  Rows: 1000
  Cleaning columns: ['persona_alpha', 'persona_beta', 'persona_gamma']

  persona_alpha:
    Encoding artifacts : 0 → 0
    Options dict rows  : 1000  → 0
  persona_beta:
    Encoding artifacts : 0 → 0
    Options dict rows  : 883  → 0
  persona_gamma:
    Encoding artifacts : 0 → 0
    Options dict rows  : 963  → 0

✓ Saved: medqa_1000_with_personas_FINAL.csv


In [7]:
"""
fix_personas.py
---------------
Two fixes applied to persona_alpha, persona_beta, persona_gamma columns:

1. Remove trailing "OPTIONS:" / "Options:" lines (with or without trailing whitespace/newlines)
2. Fix mojibake encoding artifacts (â€", â€™, etc.)
"""

import re
import pandas as pd

INPUT_CSV  = "medqa_1000_with_personas_FINAL.csv"
OUTPUT_CSV = "medqa_1000_with_personas_FINAL.csv"  # overwrite in place

PERSONA_COLS = ["persona_alpha", "persona_beta", "persona_gamma"]

# ── Encoding fix map ──────────────────────────────────────────────────────────
# These are the most common mojibake artifacts from UTF-8 decoded as latin-1
ENCODING_FIXES = {
    "\u00e2\u0080\u0099": "\u2019",   # right single quotation mark
    "\u00e2\u0080\u0098": "\u2018",   # left single quotation mark
    "\u00e2\u0080\u009c": "\u201c",   # left double quotation mark
    "\u00e2\u0080\u009d": "\u201d",   # right double quotation mark
    "\u00e2\u0080\u0094": "\u2014",   # em dash
    "\u00e2\u0080\u0093": "\u2013",   # en dash
    "\u00e2\u0080\u00a6": "\u2026",   # ellipsis
    "\u00c2\u00b0":       "\u00b0",   # degree sign
    "\u00c3\u00a9":       "\u00e9",   # e acute
    "\u00ce\u00bc":       "\u03bc",   # mu
    "\u00c2\u00a3":       "\u00a3",   # pound sign
    "\u00c3\u00a0":       "\u00e0",   # a grave
    "\u00c3\u00a8":       "\u00e8",   # e grave
    "\u00c3\u00ac":       "\u00ec",   # i grave
    "\u00c3\u00b2":       "\u00f2",   # o grave
    "\u00c3\u00b9":       "\u00f9",   # u grave
}

def fix_encoding(text: str) -> str:
    if not isinstance(text, str):
        return text
    # Attempt round-trip fix first (catches most cases in one shot)
    try:
        fixed = text.encode("latin-1").decode("utf-8")
        return fixed
    except (UnicodeDecodeError, UnicodeEncodeError):
        pass
    # Manual fallback for residual artifacts
    for bad, good in ENCODING_FIXES.items():
        text = text.replace(bad, good)
    return text


def remove_trailing_options(text: str) -> str:
    """
    Remove any trailing OPTIONS: / Options: block at the end of a persona string.
    Handles:
      - "OPTIONS:\n"
      - "Options:\n"
      - "OPTIONS:" with no newline (end of string)
      - Multiple blank lines before the OPTIONS: marker
    """
    if not isinstance(text, str):
        return text
    # Strip trailing whitespace/newlines first, then remove OPTIONS: suffix
    text = text.rstrip()
    # Remove OPTIONS: or Options: at the very end (case-insensitive)
    text = re.sub(r'\n+[Oo][Pp][Tt][Ii][Oo][Nn][Ss]:\s*$', '', text)
    # Also catch it if it's the only thing on the last line with no preceding newline
    text = re.sub(r'^[Oo][Pp][Tt][Ii][Oo][Nn][Ss]:\s*$', '', text, flags=re.MULTILINE)
    return text.rstrip()


def clean_persona(text: str) -> str:
    text = fix_encoding(text)
    text = remove_trailing_options(text)
    return text


def main():
    print(f"Loading {INPUT_CSV}...")
    df = pd.read_csv(INPUT_CSV)
    print(f"Loaded {len(df)} rows, {len(df.columns)} columns")

    for col in PERSONA_COLS:
        if col not in df.columns:
            print(f"  Skipping {col} — not in CSV")
            continue

        before_nulls = df[col].isna().sum()
        df[col] = df[col].apply(clean_persona)
        after_nulls = df[col].isna().sum()

        # Count how many had OPTIONS: removed
        options_removed = df[col].apply(
            lambda x: bool(isinstance(x, str) and
                           re.search(r'[Oo][Pp][Tt][Ii][Oo][Nn][Ss]:\s*$',
                                     x.rstrip()))
        ).sum()

        print(f"  {col}: cleaned. Nulls before={before_nulls}, after={after_nulls}")

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved cleaned CSV to {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

Loading medqa_1000_with_personas_FINAL.csv...
Loaded 1000 rows, 9 columns
  persona_alpha: cleaned. Nulls before=0, after=0
  persona_beta: cleaned. Nulls before=0, after=0
  persona_gamma: cleaned. Nulls before=0, after=0

Saved cleaned CSV to medqa_1000_with_personas_FINAL.csv


In [10]:
import pandas as pd

df = pd.read_csv("medqa_1000_with_personas_FINAL.csv")

# Show first 5 rows, only the persona columns + question_id
cols = ["question_id", "persona_alpha", "persona_beta", "persona_gamma"]
pd.set_option("display.max_colwidth", 300)
df[cols].head(5)

,question_id,persona_alpha,persona_beta,persona_gamma
0,MQ_0001,"A 52-year-old male presents for evaluation of acute malaise. Since yesterday, I have experienced nausea, emesis, diarrhea, generalized myalgias, rhinorrhea, and arthralgias. My past medical history is significant for obesity, chronic obstructive pulmonary disease, chronic lumbar pain, and fibrom...","I'm terrified they're gonna fire me at the school for missing another day of cleaning duty, but I can't even stand up straight without feeling like I'm gonna lose it. I was in here just last week to get my pills refilled and I can't afford to be back here paying another copay when I'm already sh...","My eldest daughter placed her hand upon my shoulder this morning and observed that my spirit has lost its color, and she told me I must seek guidance here immediately. I am a 52-year-old man, and I come to you because since yesterday, my internal state has become deeply disturbed. My stomach is ..."
1,MQ_0002,"A 4-year-old male presents with a several-month history of progressive lower extremity weakness. Previously active, he now exhibits exercise intolerance, requiring rest after 15 minutes of play. He describes his legs as ""sleepy."" His mother notes frequent tripping, necessitating the removal of h...","I'm terrified this is gonna keep me from picking up my extra shifts at the warehouse, because if I can't work, we're gonna be underwater on rent by next month. My four-year-old boy, he just ain't right. He used to be outside with the neighbors' kids all day long, but these past few months, he's ...","The quiet onset of this change began in a subtle manner, though my husband and I have watched it unfold with growing sorrow. My four-year-old son, who once possessed a spirit that thrived in the open air, playing with the neighbors for hours, now finds that his vitality deserts him after only fi..."
2,MQ_0003,"I am a 46-year-old female presenting with a five-month history of diffuse myalgia and arthralgia. The pain is migratory and fluctuates in intensity. I report morning joint stiffness and persistent daytime fatigue, which I attribute to sleep fragmentation. Additionally, I have developed paresthes...","If I lose another shift at the warehouse, I'm not gonna be able to cover the rent, but I can't keep humping these crates when my body is falling apart like this. I've been trying to just push through it for five months now, loading up on store-brand Tylenol and just hoping it'd quit, but it's on...","My eldest daughter, who prepares the morning cumin-seed infusions for my quiet hours, grew distressed watching me struggle to rise from my bedding; she insisted I bring this heaviness to your attention. For five months, my muscles have held a deep, persistent complaint that shifts its residence ..."
3,MQ_0004,"A 65-year-old male presents to his primary care physician for a several-month history of progressive behavioral changes. The patient initially exhibited disinhibition, characterized by inappropriate social conduct including profanity and tactile impulsivity. This has since progressed to cognitiv...","I've been dosing him with St. John's Wort tea and some heavy-duty melatonin from the dollar store for months, hoping he'd just settle down, but it ain't doing a damn thing. I had to drag him in here because I'm losing hours at the warehouse and I can't keep watching him like a hawk while I'm try...","My eldest son, who monitors my life with great care, insisted that I present myself here today. For weeks, he has prepared a decoction of bitter melon and turmeric, believing my spirit was simply restless, but the concoction provided no peace. \n\nI am a 65-year-old man, and my own mind has beco..."
4,MQ_0005,"I am a 31-year-old male, currently two days post-admission for substance-induced psychosis. Over the past 8 hours, I have experienced episodic cervical rigidity and pain. These episodes persist for approximately 25 minutes, characterized by involuntary right-sided tort